## UCI Diabetes 130-US hospitals dataset Experiment

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')

output_dir = './thesis_figures/'
os.makedirs(output_dir, exist_ok=True)

SEEDS = [42, 43, 44]
metrics = ['AUROC', 'AUPRC', 'Brier Score']
all_results = []


## Dataset

The UCI Diabetes 130-US Hospitals dataset covers 101,766 inpatient encounters for diabetic patients across 130 US hospitals between 1999 and 2008. Each row represents one hospital visit, characterised by demographic information (age, race, gender), clinical variables (diagnoses, admission type, time in hospital, lab results), and medication records. The binary target is early readmission: encounters with readmission within 30 days are labelled 1; all others are labelled 0. Encounters with `admission_source_id == 17` are held out as the shifted test set, representing a distinct admission subgroup that serves as the natural distribution shift.

In [2]:
df = pd.read_csv('./raw_data/diabetic_data.csv')
print(f"DataFrame shape before processing: {df.shape}")
df.head()


DataFrame shape before processing: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Dataset Cleaning

1. Replace '?' with NaN
2. Target: readmitted `<30` days → 1, otherwise → 0
3. Define binary target variable `y`
4. Drop columns with too many missing values, irrelevant columns, and leakage columns


In [3]:
df.replace('?', None, inplace=True)

df['readmitted_raw'] = df['readmitted']
df['y'] = df['readmitted_raw'].map({'<30': 1, '>30': 0, 'NO': 0}).astype(int)

columns_to_drop = ['encounter_id', 'patient_nbr', 'weight', 'max_glu_serum', 'A1Cresult',
                   'payer_code', 'medical_specialty', 'readmitted', 'readmitted_raw']
df.drop(columns=columns_to_drop, inplace=True)

print(f"DataFrame shape after cleaning: {df.shape}")
df.head()


DataFrame shape after cleaning: (101766, 43)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,y
0,Caucasian,Female,[0-10),6,25,1,1,41,0,1,...,No,No,No,No,No,No,No,No,No,0
1,Caucasian,Female,[10-20),1,1,7,3,59,0,18,...,No,Up,No,No,No,No,No,Ch,Yes,0
2,AfricanAmerican,Female,[20-30),1,1,7,2,11,5,13,...,No,No,No,No,No,No,No,No,Yes,0
3,Caucasian,Male,[30-40),1,1,7,2,44,1,16,...,No,Up,No,No,No,No,No,Ch,Yes,0
4,Caucasian,Male,[40-50),1,1,7,1,51,0,8,...,No,Steady,No,No,No,No,No,Ch,Yes,0


## Data Splitting

Shifted test set: all rows where `admission_source_id == 17`.

Everything else is split into:
- **In-domain train** (70%)
- **In-domain validation** (15%)
- **In-domain test** (15%)

Splitting is parameterised by `random_state` so each seed produces a different but reproducible split.


In [4]:
def get_X_y(df_input):
    """Separate features (X) and target (y) from a DataFrame."""
    if 'y' in df_input.columns:
        return df_input.drop(columns=['y']), df_input['y']
    raise ValueError("DataFrame must contain a 'y' column.")

def data_split_function(df, random_state):
    """Split data into train, validation, in-domain test, and shifted test sets."""
    df_shifted = df[df['admission_source_id'] == 17].copy()
    df_remaining = df[df['admission_source_id'] != 17].copy()

    df_train, df_temp = train_test_split(
        df_remaining, test_size=0.30, stratify=df_remaining['y'], random_state=random_state
    )
    df_val, df_test = train_test_split(
        df_temp, test_size=0.50, stratify=df_temp['y'], random_state=random_state
    )

    X_train, y_train = get_X_y(df_train)
    X_val, y_val = get_X_y(df_val)
    X_test, y_test = get_X_y(df_test)
    X_shifted, y_shifted = get_X_y(df_shifted)

    print(f"  Train: {X_train.shape}, Val: {X_val.shape}, Test ID: {X_test.shape}, Shifted: {X_shifted.shape}")
    return X_train, y_train, X_val, y_val, X_test, y_test, X_shifted, y_shifted


In [5]:
def evaluate_model(model, X_test, y_test):
    """Evaluate a trained model, returning AUROC, AUPRC, and Brier Score."""
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    auroc = roc_auc_score(y_test, y_pred_proba)
    auprc = average_precision_score(y_test, y_pred_proba)
    brier = brier_score_loss(y_test, y_pred_proba)
    return auroc, auprc, brier


def train_models(X_train, y_train, random_state):
    """Re-initialise and fit LR and LightGBM pipelines from scratch."""
    numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

    cat_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor_lr = ColumnTransformer(
        transformers=[
            ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                                    ('scaler', StandardScaler(with_mean=False))]), numerical_cols),
            ('cat', cat_pipeline, categorical_cols)
        ],
        remainder='drop'
    )
    preprocessor_lgbm = ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), numerical_cols),
            ('cat', clone(cat_pipeline), categorical_cols)
        ],
        remainder='drop'
    )

    pipeline_lr = Pipeline(steps=[
        ('preprocessor', preprocessor_lr),
        ('classifier', LogisticRegression(solver='liblinear', random_state=random_state, class_weight='balanced'))
    ])
    pipeline_lr.fit(X_train, y_train)

    pipeline_lgbm = Pipeline(steps=[
        ('preprocessor', preprocessor_lgbm),
        ('classifier', LGBMClassifier(random_state=random_state, verbose=-1, class_weight='balanced'))
    ])
    pipeline_lgbm.fit(X_train, y_train)

    return pipeline_lr, pipeline_lgbm


print('evaluate_model and train_models defined.')


evaluate_model and train_models defined.


## Data Perturbation Helpers


In [ ]:
def introduce_label_noise(y, noise_rate, random_state):
    """Introduce noise into training labels by flipping a fraction of them."""
    y_noisy = y.copy()
    if noise_rate == 0:
        return y_noisy
    num_flips = int(len(y_noisy) * noise_rate)
    np.random.seed(random_state)
    flip_indices = np.random.choice(y_noisy.index, size=num_flips, replace=False)
    y_noisy.loc[flip_indices] = 1 - y_noisy.loc[flip_indices]
    return y_noisy


def introduce_mcar_missingness(df, columns_to_modify, missing_rate, random_state):
    """Introduce Missing Completely At Random (MCAR) missingness."""
    df_mcar = df.copy()
    np.random.seed(random_state)
    for col in columns_to_modify:
        if col in df_mcar.columns:
            non_nan_idx = df_mcar[col].dropna().index
            num_nan = min(int(len(df_mcar[col]) * missing_rate), len(non_nan_idx))
            if num_nan > 0:
                fill = None if df_mcar[col].dtype == object else np.nan
                df_mcar.loc[np.random.choice(non_nan_idx, size=num_nan, replace=False), col] = fill
    return df_mcar


def introduce_mar_missingness(df, columns_to_modify, missing_rate, condition_column, condition_values, random_state):
    """Introduce Missing At Random (MAR) missingness conditioned on a column's values."""
    df_mar = df.copy()
    np.random.seed(random_state)
    cond_idx = df_mar[df_mar[condition_column].isin(condition_values)].index
    if len(cond_idx) == 0 and missing_rate > 0:
        print(f"Warning: No rows match condition {condition_column}={condition_values}. Skipping MAR.")
        return df_mar
    for col in columns_to_modify:
        if col in df_mar.columns:
            target_idx = df_mar.loc[cond_idx, col].dropna().index
            num_nan = int(len(target_idx) * missing_rate)
            if num_nan > 0:
                fill = None if df_mar[col].dtype == object else np.nan
                df_mar.loc[np.random.choice(target_idx, size=num_nan, replace=False), col] = fill
    return df_mar


def blank_out_features(df_input, cols_to_blank):
    """Set specified feature columns to missing (simulate feature unavailability)."""
    df_blanked = df_input.copy()
    for col in cols_to_blank:
        if col in df_blanked.columns:
            if df_blanked[col].dtype == object:
                df_blanked[col] = None  
            else:
                df_blanked[col] = np.nan  
    return df_blanked


medication_related_keywords = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]
all_feature_cols = [c for c in df.columns if c != 'y']
diagnoses_cols = [col for col in all_feature_cols if 'diag_' in col]
medications_cols = [col for col in all_feature_cols if col in medication_related_keywords]
feature_groups = {'Diagnoses': diagnoses_cols, 'Medications': medications_cols}

print(f"Diagnoses columns: {diagnoses_cols}")
print(f"Medications columns: {medications_cols}")


Diagnoses columns: ['diag_1', 'diag_2', 'diag_3']
Medications columns: ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']


In [7]:
all_results = []  # Reset accumulator before running experiments

## Baseline Experiment

Both models are trained on **clean in-domain data** with no perturbations. Results on the in-domain test set and the shifted test set (`admission_source_id == 17`) establish the performance reference for all subsequent comparisons.

The trained models from this section are reused in feature-availability shift, where blanking is applied only at evaluation time.

In [8]:
def run_baseline_experiment(seed, X_train, y_train, X_test, y_test, X_shifted, y_shifted):
    """Baseline: train on clean data, evaluate in-domain and shifted."""
    print(f"--- Baseline | Seed {seed} ---")
    pipeline_lr, pipeline_lgbm = train_models(X_train, y_train, seed)

    for model_name, pipeline in [('Logistic Regression', pipeline_lr), ('LightGBM', pipeline_lgbm)]:
        auroc_id, auprc_id, brier_id = evaluate_model(pipeline, X_test, y_test)
        auroc_sh, auprc_sh, brier_sh = evaluate_model(pipeline, X_shifted, y_shifted)
        base = {'Experiment': 'Baseline', 'Condition': 'None', 'Noise Rate': 0,
                'Missingness Type': 'None', 'Missing Rate': 0, 'Feature Group Blanked': 'None',
                'Model': model_name, 'Random Seed': seed}
        all_results.append({**base, 'Test Set': 'In-domain Test',
                             'AUROC': auroc_id, 'AUPRC': auprc_id, 'Brier Score': brier_id})
        all_results.append({**base, 'Test Set': 'Shifted Test',
                             'AUROC': auroc_sh, 'AUPRC': auprc_sh, 'Brier Score': brier_sh})

    return pipeline_lr, pipeline_lgbm

In [9]:
baseline_pipelines = {} 

for seed in SEEDS:
    print(f"\n===== Baseline | Seed {seed} =====")
    X_train, y_train, X_val, y_val, X_test, y_test, X_shifted, y_shifted = \
        data_split_function(df, seed)
    pipeline_lr, pipeline_lgbm = run_baseline_experiment(
        seed, X_train, y_train, X_test, y_test, X_shifted, y_shifted)
    baseline_pipelines[seed] = (pipeline_lr, pipeline_lgbm, X_test, y_test, X_shifted, y_shifted)

# Mini-summary
_base_df = pd.DataFrame(all_results)
_base_df = _base_df[_base_df['Experiment'] == 'Baseline']
print("\nBaseline mean AUROC across seeds:")
print(_base_df.groupby(['Model', 'Test Set'])['AUROC'].mean().round(4).to_string())


===== Baseline | Seed 42 =====
  Train: (66489, 42), Val: (14248, 42), Test ID: (14248, 42), Shifted: (6781, 42)
--- Baseline | Seed 42 ---

===== Baseline | Seed 43 =====
  Train: (66489, 42), Val: (14248, 42), Test ID: (14248, 42), Shifted: (6781, 42)
--- Baseline | Seed 43 ---

===== Baseline | Seed 44 =====
  Train: (66489, 42), Val: (14248, 42), Test ID: (14248, 42), Shifted: (6781, 42)
--- Baseline | Seed 44 ---

Baseline mean AUROC across seeds:
Model                Test Set      
LightGBM             In-domain Test    0.6736
                     Shifted Test      0.6586
Logistic Regression  In-domain Test    0.6284
                     Shifted Test      0.6255


## Label Noise Experiment

Training labels are randomly flipped at rates of **0%, 5%, 10%, and 20%**. The 0% condition replicates clean labels and serves as an internal anchor. This section tests how sensitive each model's generalisation gap is to progressively corrupted supervision signals.

In [10]:
def run_label_noise_experiment(seed, X_train, y_train, X_test, y_test, X_shifted, y_shifted):
    """Label noise sensitivity: vary noise rate at 0%, 5%, 10%, 20%."""
    print(f"--- Label Noise | Seed {seed} ---")
    for noise_rate in [0.0, 0.05, 0.10, 0.20]:
        print(f"  Noise rate: {noise_rate*100:.0f}%")
        y_noisy = introduce_label_noise(y_train, noise_rate, random_state=seed)
        pipeline_lr, pipeline_lgbm = train_models(X_train, y_noisy, seed)

        for model_name, pipeline in [('Logistic Regression', pipeline_lr), ('LightGBM', pipeline_lgbm)]:
            auroc_id, auprc_id, brier_id = evaluate_model(pipeline, X_test, y_test)
            auroc_sh, auprc_sh, brier_sh = evaluate_model(pipeline, X_shifted, y_shifted)
            base = {'Experiment': 'Label Noise', 'Condition': f'{noise_rate*100:.0f}% Noise',
                    'Noise Rate': noise_rate, 'Missingness Type': 'None', 'Missing Rate': 0,
                    'Feature Group Blanked': 'None', 'Model': model_name, 'Random Seed': seed}
            all_results.append({**base, 'Test Set': 'In-domain Test',
                                 'AUROC': auroc_id, 'AUPRC': auprc_id, 'Brier Score': brier_id})
            all_results.append({**base, 'Test Set': 'Shifted Test',
                                 'AUROC': auroc_sh, 'AUPRC': auprc_sh, 'Brier Score': brier_sh})

In [11]:
_noise_start = len(all_results)

for seed in SEEDS:
    print(f"\n===== Label Noise | Seed {seed} =====")
    X_train, y_train, X_val, y_val, X_test, y_test, X_shifted, y_shifted = \
        data_split_function(df, seed)
    run_label_noise_experiment(seed, X_train, y_train, X_test, y_test, X_shifted, y_shifted)

# Mini-summary
_noise_df = pd.DataFrame(all_results[_noise_start:])
print("\nLabel Noise mean AUROC (Shifted Test) across seeds:")
print(_noise_df[_noise_df['Test Set'] == 'Shifted Test']
      .groupby(['Model', 'Condition'])['AUROC'].mean().round(4).to_string())


===== Label Noise | Seed 42 =====
  Train: (66489, 42), Val: (14248, 42), Test ID: (14248, 42), Shifted: (6781, 42)
--- Label Noise | Seed 42 ---
  Noise rate: 0%
  Noise rate: 5%
  Noise rate: 10%
  Noise rate: 20%

===== Label Noise | Seed 43 =====
  Train: (66489, 42), Val: (14248, 42), Test ID: (14248, 42), Shifted: (6781, 42)
--- Label Noise | Seed 43 ---
  Noise rate: 0%
  Noise rate: 5%
  Noise rate: 10%
  Noise rate: 20%

===== Label Noise | Seed 44 =====
  Train: (66489, 42), Val: (14248, 42), Test ID: (14248, 42), Shifted: (6781, 42)
--- Label Noise | Seed 44 ---
  Noise rate: 0%
  Noise rate: 5%
  Noise rate: 10%
  Noise rate: 20%

Label Noise mean AUROC (Shifted Test) across seeds:
Model                Condition
LightGBM             0% Noise     0.6586
                     10% Noise    0.6432
                     20% Noise    0.6097
                     5% Noise     0.6542
Logistic Regression  0% Noise     0.6255
                     10% Noise    0.6101
                   

## Missingness Experiment

Three missingness patterns are injected into training features (`time_in_hospital`, `num_lab_procedures`, `race`, `diag_1`) at rates of **0%, 10%, and 20%**:
- **MCAR** (Missing Completely At Random): values are blanked randomly, independent of any other feature.
- **MAR_Female** (Missing At Random): values are blanked only for female patients, simulating a gender-linked data collection gap.
- **MAR_Age_Older** (Missing At Random): values are blanked only for patients aged 70–90, simulating a gap for older respondents.

In [12]:
def run_missingness_experiment(seed, X_train, y_train, X_test, y_test, X_shifted, y_shifted):
    """Missingness injection: MCAR and MAR at 0%, 10%, 20%."""
    print(f"--- Missingness | Seed {seed} ---")
    missing_rates = [0.0, 0.10, 0.20]
    columns_for_missingness = ['time_in_hospital', 'num_lab_procedures', 'race', 'diag_1']
    missingness_scenarios = [
        {'type': 'MCAR', 'condition': None,                                              'label': 'MCAR'},
        {'type': 'MAR',  'condition': {'column': 'gender',  'values': ['Female']},       'label': 'MAR_Female'},
        {'type': 'MAR',  'condition': {'column': 'age',     'values': ['[70-80)', '[80-90)']}, 'label': 'MAR_Age_Older'},
    ]

    for scenario in missingness_scenarios:
        label = scenario['label']
        for rate in missing_rates:
            print(f"  {label} @ {rate*100:.0f}%")
            if scenario['type'] == 'MCAR':
                X_train_noisy = introduce_mcar_missingness(X_train, columns_for_missingness, rate, seed)
            else:
                if rate == 0:
                    X_train_noisy = X_train.copy()
                else:
                    cond = scenario['condition']
                    actual_vals = [v for v in cond['values'] if v in X_train[cond['column']].unique()]
                    if actual_vals:
                        X_train_noisy = introduce_mar_missingness(
                            X_train, columns_for_missingness, rate,
                            cond['column'], actual_vals, seed
                        )
                    else:
                        X_train_noisy = X_train.copy()

            pipeline_lr, pipeline_lgbm = train_models(X_train_noisy, y_train, seed)

            for model_name, pipeline in [('Logistic Regression', pipeline_lr), ('LightGBM', pipeline_lgbm)]:
                auroc_id, auprc_id, brier_id = evaluate_model(pipeline, X_test, y_test)
                auroc_sh, auprc_sh, brier_sh = evaluate_model(pipeline, X_shifted, y_shifted)
                base = {'Experiment': 'Missingness', 'Condition': label, 'Noise Rate': 0,
                        'Missingness Type': label, 'Missing Rate': rate, 'Feature Group Blanked': 'None',
                        'Model': model_name, 'Random Seed': seed}
                all_results.append({**base, 'Test Set': 'In-domain Test',
                                     'AUROC': auroc_id, 'AUPRC': auprc_id, 'Brier Score': brier_id})
                all_results.append({**base, 'Test Set': 'Shifted Test',
                                     'AUROC': auroc_sh, 'AUPRC': auprc_sh, 'Brier Score': brier_sh})

In [13]:
_miss_start = len(all_results)

for seed in SEEDS:
    print(f"\n===== Missingness | Seed {seed} =====")
    X_train, y_train, X_val, y_val, X_test, y_test, X_shifted, y_shifted = \
        data_split_function(df, seed)
    run_missingness_experiment(seed, X_train, y_train, X_test, y_test, X_shifted, y_shifted)

_miss_df = pd.DataFrame(all_results[_miss_start:])
print("\nMissingness mean AUROC (Shifted Test) across seeds:")
print(_miss_df[_miss_df['Test Set'] == 'Shifted Test']
      .groupby(['Model', 'Missingness Type', 'Missing Rate'])['AUROC'].mean().round(4).to_string())


===== Missingness | Seed 42 =====
  Train: (66489, 42), Val: (14248, 42), Test ID: (14248, 42), Shifted: (6781, 42)
--- Missingness | Seed 42 ---
  MCAR @ 0%
  MCAR @ 10%
  MCAR @ 20%
  MAR_Female @ 0%
  MAR_Female @ 10%
  MAR_Female @ 20%
  MAR_Age_Older @ 0%
  MAR_Age_Older @ 10%
  MAR_Age_Older @ 20%

===== Missingness | Seed 43 =====
  Train: (66489, 42), Val: (14248, 42), Test ID: (14248, 42), Shifted: (6781, 42)
--- Missingness | Seed 43 ---
  MCAR @ 0%
  MCAR @ 10%
  MCAR @ 20%
  MAR_Female @ 0%
  MAR_Female @ 10%
  MAR_Female @ 20%
  MAR_Age_Older @ 0%
  MAR_Age_Older @ 10%
  MAR_Age_Older @ 20%

===== Missingness | Seed 44 =====
  Train: (66489, 42), Val: (14248, 42), Test ID: (14248, 42), Shifted: (6781, 42)
--- Missingness | Seed 44 ---
  MCAR @ 0%
  MCAR @ 10%
  MCAR @ 20%
  MAR_Female @ 0%
  MAR_Female @ 10%
  MAR_Female @ 20%
  MAR_Age_Older @ 0%
  MAR_Age_Older @ 10%
  MAR_Age_Older @ 20%

Missingness mean AUROC (Shifted Test) across seeds:
Model                Missingn

## Feature-Availability Shift Experiment

Two clinical feature groups are blanked entirely **at evaluation time** (both in-domain and shifted test sets). Models are trained on complete, clean data; the blanking simulates a deployment scenario where certain data sources become unavailable.

Group and Columns blanked.


Diagnoses: diag_1, diag_2, diag_3 

Medications: metformin, repaglinide, nateglinide, chlorpropamide, glimepiride, acetohexamide, glipizide, glyburide, tolbutamide, pioglitazone, rosiglitazone, acarbose, miglitol, troglitazone, tolazamide, examide, citoglipton, insulin, and combination drugs

The baseline-trained models from Section A are reused here — no retraining is needed.

In [14]:
def run_feature_shift_experiment(seed, X_test, y_test, X_shifted, y_shifted,
                                  pipeline_lr, pipeline_lgbm):
    """Feature shift: blank out feature groups at test time using baseline-trained models."""
    print(f"--- Feature Shift | Seed {seed} ---")
    for group_name, cols_to_blank in feature_groups.items():
        print(f"  Blanking: {group_name}")
        X_test_blanked    = blank_out_features(X_test, cols_to_blank)
        X_shifted_blanked = blank_out_features(X_shifted, cols_to_blank)

        for model_name, pipeline in [('Logistic Regression', pipeline_lr), ('LightGBM', pipeline_lgbm)]:
            auroc_id, auprc_id, brier_id = evaluate_model(pipeline, X_test_blanked, y_test)
            auroc_sh, auprc_sh, brier_sh = evaluate_model(pipeline, X_shifted_blanked, y_shifted)
            base = {'Experiment': 'Feature Shift', 'Condition': group_name, 'Noise Rate': 0,
                    'Missingness Type': 'None', 'Missing Rate': 0, 'Feature Group Blanked': group_name,
                    'Model': model_name, 'Random Seed': seed}
            all_results.append({**base, 'Test Set': 'In-domain Test',
                                 'AUROC': auroc_id, 'AUPRC': auprc_id, 'Brier Score': brier_id})
            all_results.append({**base, 'Test Set': 'Shifted Test',
                                 'AUROC': auroc_sh, 'AUPRC': auprc_sh, 'Brier Score': brier_sh})

In [15]:
_feat_start = len(all_results)

for seed in SEEDS:
    print(f"\n===== Feature Shift | Seed {seed} =====")
    pipeline_lr, pipeline_lgbm, X_test, y_test, X_shifted, y_shifted = baseline_pipelines[seed]
    run_feature_shift_experiment(seed, X_test, y_test, X_shifted, y_shifted, pipeline_lr, pipeline_lgbm)

# Mini-summary
_feat_df = pd.DataFrame(all_results[_feat_start:])
print("\nFeature Shift mean AUROC (Shifted Test) across seeds:")
print(_feat_df[_feat_df['Test Set'] == 'Shifted Test']
      .groupby(['Model', 'Feature Group Blanked'])['AUROC'].mean().round(4).to_string())

print("\n\nAll experiments complete.")


===== Feature Shift | Seed 42 =====
--- Feature Shift | Seed 42 ---
  Blanking: Diagnoses
  Blanking: Medications

===== Feature Shift | Seed 43 =====
--- Feature Shift | Seed 43 ---
  Blanking: Diagnoses
  Blanking: Medications

===== Feature Shift | Seed 44 =====
--- Feature Shift | Seed 44 ---
  Blanking: Diagnoses
  Blanking: Medications

Feature Shift mean AUROC (Shifted Test) across seeds:
Model                Feature Group Blanked
LightGBM             Diagnoses                0.6598
                     Medications              0.6520
Logistic Regression  Diagnoses                0.6315
                     Medications              0.6231


All experiments complete.


## Aggregate and Summarize Results

1. Build `results_df` from raw results
2. Compute the generalization gap (Shifted - In-domain) per run and append to `results_df`
3. Build `summary_df` with mean ± std across seeds


In [16]:
results_df = pd.DataFrame(all_results)
print(f"Raw results shape: {results_df.shape}")
results_df.head()


Raw results shape: (192, 12)


,Experiment,Condition,Noise Rate,Missingness Type,Missing Rate,Feature Group Blanked,Model,Random Seed,Test Set,AUROC,AUPRC,Brier Score
0,Baseline,None,0.0,None,0.0,None,Logistic Regression,42,In-domain Test,0.622352,0.183679,0.227618
1,Baseline,None,0.0,None,0.0,None,Logistic Regression,42,Shifted Test,0.621996,0.183928,0.223621
2,Baseline,None,0.0,None,0.0,None,LightGBM,42,In-domain Test,0.674454,0.217205,0.212715
3,Baseline,None,0.0,None,0.0,None,LightGBM,42,Shifted Test,0.653152,0.200395,0.210293
4,Baseline,None,0.0,None,0.0,None,Logistic Regression,43,In-domain Test,0.635468,0.189298,0.224674


In [17]:
# Compute generalization gap (Shifted - In-domain) per run and append
gap_cols = ['AUROC', 'AUPRC', 'Brier Score']
group_keys = ['Experiment', 'Condition', 'Noise Rate', 'Missingness Type',
              'Missing Rate', 'Feature Group Blanked', 'Model', 'Random Seed']
gap_rows = []

for _, group in results_df.groupby(group_keys):
    id_row = group[group['Test Set'] == 'In-domain Test']
    sh_row = group[group['Test Set'] == 'Shifted Test']
    if not id_row.empty and not sh_row.empty:
        gap = id_row.iloc[0].to_dict()
        gap['Test Set'] = 'Generalization Gap (Shifted - In-domain)'
        for col in gap_cols:
            gap[col] = sh_row.iloc[0][col] - id_row.iloc[0][col]
        gap_rows.append(gap)

results_df = pd.concat([results_df, pd.DataFrame(gap_rows)], ignore_index=True)
print(f"Results shape (with gaps): {results_df.shape}")


Results shape (with gaps): (288, 12)


In [18]:
summary_df = results_df.groupby(
    ['Experiment', 'Condition', 'Noise Rate', 'Missingness Type',
     'Missing Rate', 'Feature Group Blanked', 'Model', 'Test Set']
).agg(
    mean_AUROC=('AUROC', 'mean'),       std_AUROC=('AUROC', 'std'),
    mean_AUPRC=('AUPRC', 'mean'),       std_AUPRC=('AUPRC', 'std'),
    mean_Brier_Score=('Brier Score', 'mean'), std_Brier_Score=('Brier Score', 'std')
).reset_index()

print(f"Summary shape: {summary_df.shape}")
summary_df.head()


Summary shape: (96, 14)


,Experiment,Condition,Noise Rate,Missingness Type,Missing Rate,Feature Group Blanked,Model,Test Set,mean_AUROC,std_AUROC,mean_AUPRC,std_AUPRC,mean_Brier_Score,std_Brier_Score
0,Baseline,None,0.0,None,0.0,None,LightGBM,Generalization Gap (Shifted - In-domain),-0.014998,0.005460,-0.012039,0.009281,0.007183,0.011344
1,Baseline,None,0.0,None,0.0,None,LightGBM,In-domain Test,0.673558,0.002011,0.215421,0.004207,0.212593,0.000723
2,Baseline,None,0.0,None,0.0,None,LightGBM,Shifted Test,0.658559,0.005080,0.203382,0.005101,0.219776,0.010789
3,Baseline,None,0.0,None,0.0,None,Logistic Regression,Generalization Gap (Shifted - In-domain),-0.002913,0.002469,-0.001143,0.001887,0.003494,0.012594
4,Baseline,None,0.0,None,0.0,None,Logistic Regression,In-domain Test,0.628408,0.006616,0.184307,0.004708,0.226046,0.001482


## Save Results


In [19]:
results_df.to_csv(os.path.join(output_dir, 'all_experiments_raw_results.csv'), index=False)
summary_df.to_csv(os.path.join(output_dir, 'all_experiments_summary.csv'), index=False)
print(f"Saved to {output_dir}")


Saved to ./thesis_figures/


## Generate Plots

- `plot_metrics_line_chart`: mean metric vs. condition with ±1 std band, for In-domain and Shifted test sets
- `plot_gap_line_chart`: generalization gap (Shifted − In-domain) vs. condition
- `plot_gap_bar_chart`: bar chart of generalization gap by feature group


In [20]:
def _mean_std_cols(metric):
    """Return summary_df column names for a given metric."""
    key = metric.replace(' ', '_')
    return f'mean_{key}', f'std_{key}'


def plot_baseline_bar_chart(summary_df, output_dir, file_prefix='baseline'):
    """Grouped bar chart: In-domain vs Shifted performance per model."""
    plot_df = summary_df[
        (summary_df['Experiment'] == 'Baseline') &
        (summary_df['Test Set'].isin(['In-domain Test', 'Shifted Test']))
    ].copy()

    test_sets = ['In-domain Test', 'Shifted Test']
    model_names = plot_df['Model'].unique()
    x = np.arange(len(model_names))
    width = 0.35
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    for metric in metrics:
        mean_col, std_col = _mean_std_cols(metric)
        fig, ax = plt.subplots(figsize=(9, 6))
        for i, ts in enumerate(test_sets):
            ts_df = plot_df[plot_df['Test Set'] == ts]
            means = [ts_df[ts_df['Model'] == m][mean_col].values[0]
                     if not ts_df[ts_df['Model'] == m].empty else 0 for m in model_names]
            stds  = [ts_df[ts_df['Model'] == m][std_col].fillna(0).values[0]
                     if not ts_df[ts_df['Model'] == m].empty else 0 for m in model_names]
            ax.bar(x + i * width, means, width, yerr=stds, capsize=5, label=ts, color=colors[i])
        ax.set_title(f'Baseline {metric}: In-domain vs Shifted', fontsize=14)
        ax.set_xlabel('Model', fontsize=12)
        ax.set_ylabel(metric, fontsize=12)
        ax.set_xticks(x + width / 2)
        ax.set_xticklabels(model_names)
        ax.legend(title='Test Set')
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{file_prefix}_{metric.lower().replace(" ", "_")}.png'), bbox_inches='tight')
        plt.close()
    print('Baseline bar charts saved.')


def plot_metrics_line_chart(summary_df, experiment_name, x_axis_column, hue_column, output_dir, file_prefix):
    """Line chart: mean metric vs. x_axis_column, split by model and test set."""
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    for metric in metrics:
        mean_col, std_col = _mean_std_cols(metric)
        fig, ax = plt.subplots(figsize=(14, 8))
        plot_df = summary_df[
            (summary_df['Experiment'] == experiment_name) &
            (summary_df['Test Set'].isin(['In-domain Test', 'Shifted Test']))
        ].copy().sort_values(by=x_axis_column)
        plot_df[std_col] = plot_df[std_col].fillna(0)

        ci = 0
        for model in plot_df['Model'].unique():
            for hue_val in plot_df[hue_column].unique():
                for ts in ['In-domain Test', 'Shifted Test']:
                    d = plot_df[(plot_df['Model'] == model) &
                                (plot_df[hue_column] == hue_val) &
                                (plot_df['Test Set'] == ts)]
                    if d.empty:
                        continue
                    c = colors[ci % len(colors)]
                    ax.plot(d[x_axis_column], d[mean_col], marker='o', linewidth=2, color=c,
                            label=f'{model} - {hue_val} - {ts}')
                    ax.fill_between(d[x_axis_column],
                                    d[mean_col] - d[std_col], d[mean_col] + d[std_col],
                                    alpha=0.1, color=c)
                    ci += 1

        ax.set_title(f'Mean {metric} vs. {x_axis_column} — {experiment_name}', fontsize=16)
        ax.set_xlabel(x_axis_column, fontsize=12)
        ax.set_ylabel(f'Mean {metric}', fontsize=12)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{file_prefix}_{metric.lower().replace(" ", "_")}_mean.png'), bbox_inches='tight')
        plt.close()
    print(f'Line charts saved: {experiment_name}')


def plot_gap_line_chart(summary_df, experiment_name, x_axis_column, hue_column, output_dir, file_prefix):
    """Line chart: generalization gap vs. x_axis_column."""
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    for metric in metrics:
        mean_col, std_col = _mean_std_cols(metric)
        fig, ax = plt.subplots(figsize=(14, 8))
        plot_df = summary_df[
            (summary_df['Experiment'] == experiment_name) &
            (summary_df['Test Set'] == 'Generalization Gap (Shifted - In-domain)')
        ].copy().sort_values(by=x_axis_column)

        if plot_df.empty:
            print(f'No gap data for {experiment_name} / {metric}.')
            plt.close()
            continue

        plot_df[std_col] = plot_df[std_col].fillna(0)
        ci = 0
        for model in plot_df['Model'].unique():
            for hue_val in plot_df[hue_column].unique():
                d = plot_df[(plot_df['Model'] == model) & (plot_df[hue_column] == hue_val)]
                if d.empty:
                    continue
                c = colors[ci % len(colors)]
                ax.plot(d[x_axis_column], d[mean_col], marker='o', linewidth=2, color=c,
                        label=f'{model} - {hue_val}')
                ax.fill_between(d[x_axis_column],
                                d[mean_col] - d[std_col], d[mean_col] + d[std_col],
                                alpha=0.1, color=c)
                ci += 1

        ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
        ax.set_title(f'Generalization Gap ({metric}) vs. {x_axis_column} — {experiment_name}', fontsize=16)
        ax.set_xlabel(x_axis_column, fontsize=12)
        ax.set_ylabel(f'{metric} Gap (Shifted - In-domain)', fontsize=12)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{file_prefix}_{metric.lower().replace(" ", "_")}_gap.png'), bbox_inches='tight')
        plt.close()
    print(f'Gap line charts saved: {experiment_name}')


def plot_gap_bar_chart(summary_df, experiment_name, category_column, output_dir, file_prefix):
    """Bar chart: generalization gap by category."""
    for metric in metrics:
        mean_col, std_col = _mean_std_cols(metric)
        fig, ax = plt.subplots(figsize=(14, 8))
        plot_df = summary_df[
            (summary_df['Experiment'] == experiment_name) &
            (summary_df['Test Set'] == 'Generalization Gap (Shifted - In-domain)')
        ].copy().sort_values(by=category_column)

        if plot_df.empty:
            print(f'No data for {experiment_name} gap bar chart / {metric}.')
            plt.close()
            continue

        plot_df[std_col] = plot_df[std_col].fillna(0)
        models = plot_df['Model'].unique()
        categories = plot_df[category_column].unique()
        x = np.arange(len(categories))
        width = 0.35

        for i, model_name in enumerate(models):
            md_df = plot_df[plot_df['Model'] == model_name]
            means = [md_df[md_df[category_column] == c][mean_col].values[0]
                     if not md_df[md_df[category_column] == c].empty else 0 for c in categories]
            stds  = [md_df[md_df[category_column] == c][std_col].values[0]
                     if not md_df[md_df[category_column] == c].empty else 0 for c in categories]
            ax.bar(x + i * width, means, width, yerr=stds, capsize=5, label=model_name)

        ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
        ax.set_title(f'{metric} Gap by {category_column} — {experiment_name}', fontsize=16)
        ax.set_xlabel(category_column, fontsize=12)
        ax.set_ylabel(f'{metric} Gap', fontsize=12)
        ax.set_xticks(x + width / 2 * (len(models) - 1))
        ax.set_xticklabels(categories, rotation=45, ha='right')
        ax.legend(title='Model')
        ax.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{file_prefix}_{metric.lower().replace(" ", "_")}_gap_bar.png'), bbox_inches='tight')
        plt.close()
    print(f'Bar charts saved: {experiment_name}')


print("Plot functions defined.")


Plot functions defined.


In [21]:
print("--- Generating Plots ---")

# Baseline
plot_baseline_bar_chart(summary_df, output_dir)

# Label Noise
plot_metrics_line_chart(summary_df, 'Label Noise', 'Noise Rate', 'Model', output_dir, 'label_noise')
plot_gap_line_chart(summary_df, 'Label Noise', 'Noise Rate', 'Model', output_dir, 'label_noise')

# Missingness
plot_metrics_line_chart(summary_df, 'Missingness', 'Missing Rate', 'Missingness Type', output_dir, 'missingness')
plot_gap_line_chart(summary_df, 'Missingness', 'Missing Rate', 'Missingness Type', output_dir, 'missingness')

# Feature Shift — include baseline gap for comparison
baseline_gap_df = summary_df[
    (summary_df['Experiment'] == 'Baseline') &
    (summary_df['Test Set'] == 'Generalization Gap (Shifted - In-domain)')
][['Model', 'mean_AUROC', 'std_AUROC', 'mean_AUPRC', 'std_AUPRC',
   'mean_Brier_Score', 'std_Brier_Score']].copy()
baseline_gap_df['Feature Group Blanked'] = 'Baseline (No Blanking)'
baseline_gap_df['Experiment'] = 'Feature Shift'
baseline_gap_df['Condition'] = 'Baseline'
baseline_gap_df['Noise Rate'] = 0.0
baseline_gap_df['Missingness Type'] = 'None'
baseline_gap_df['Missing Rate'] = 0.0
baseline_gap_df['Test Set'] = 'Generalization Gap (Shifted - In-domain)'

feature_shift_plot_df = pd.concat([
    summary_df[(summary_df['Experiment'] == 'Feature Shift') &
               (summary_df['Test Set'] == 'Generalization Gap (Shifted - In-domain)')],
    baseline_gap_df
], ignore_index=True)

plot_gap_bar_chart(feature_shift_plot_df, 'Feature Shift', 'Feature Group Blanked', output_dir, 'feature_shift')

print("All plots saved to", output_dir)


--- Generating Plots ---
Baseline bar charts saved.
Line charts saved: Label Noise
Gap line charts saved: Label Noise
Line charts saved: Missingness
Gap line charts saved: Missingness
Bar charts saved: Feature Shift
All plots saved to ./thesis_figures/
